# Qwen 27B — Kaggle-এ ফোনের ব্রাউজার থেকে চালানোর চ্যাট ওয়েবসাইট

**আগে:** Settings → Accelerator → **GPU T4 ×2**, আর **Internet ON** করুন।

সেলগুলো ওপর থেকে নিচে একটার পর একটা চালান। একদম শেষ সেল একটা লিঙ্ক দেবে — সেটা ফোনের Chrome-এ খুলুন।

- মডেল দুই T4-তে ভাগ হয়ে পুরোটা GPU মেমরিতে থাকে (`OLLAMA_SCHED_SPREAD`), CPU-তে কিছু নামে না
- Flash Attention + q8 KV cache: কম মেমরিতে দ্রুত
- ফোন থেকে ছবি/PDF/DOCX/XLSX/টেক্সট/কোড ফাইল পাঠিয়ে বিশ্লেষণ
- লিঙ্কে গোপন কী (`?k=...`) আছে, তাই লিঙ্ক না জানলে কেউ ঢুকতে পারবে না
- **নতুন:** মডেল আগে থেকেই GPU-তে লোড হয়ে পিন থাকে (কনটেক্সট অটো-বাছাই), প্রতি ২ মিনিটে গরম রাখা হয়
- **নতুন UI:** রঙিন আধুনিক ডিজাইন, বাঁ পাশের মেনুতে আগের চ্যাটের তালিকা, লেখার সময় জোর করে নিচে টানে না


## ১. GPU দেখুন + দরকারি প্যাকেজ

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install fastapi uvicorn httpx python-multipart pypdf python-docx openpyxl
!apt-get update -qq && apt-get install -y -qq zstd
!curl -fsSL https://ollama.com/install.sh | sh
!ollama --version

name, memory.total [MiB]
Tesla T4, 15360 MiB
Tesla T4, 15360 MiB
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package zstd.
(Reading database ... 130198 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%###############################################       93.9%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434

## ২. সেটিংস (দরকার হলে এখানে বদলান)

In [ ]:
import os, subprocess, shutil, time, json, urllib.request, secrets
from pathlib import Path

SOURCE_MODEL = "hf.co/JonathanColetti/Qwen3.8-27B-Uncensored-GGUF:Q4_K_M"
NUM_CTX = 32768          # 32K: দ্রুত + নিরাপদে ৩০GB-তে ধরে। পুরো GPU-তে ঢুকলে 49152/65536 চেষ্টা করতে পারেন
VISION_HELPER = "qwen2.5vl:3b"   # মডেলের নিজের ছবি-দেখার ক্ষমতা না থাকলে শুধু তখনই নামবে
NUM_CTX_FALLBACKS = [32768, 24576, 16384, 12288]   # পুরোটা GPU-তে না ঢুকলে সেল ৬ নিজেই ছোটটা চেষ্টা করবে
PORT = 7860

# /kaggle/temp-এ বেশি জায়গা থাকে; না থাকলে /kaggle/working
base = Path("/kaggle/temp") if Path("/kaggle/temp").exists() else Path("/kaggle/working")
MODELS_DIR = base / "ollama-models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

OLLAMA_URL = "http://127.0.0.1:11434"
OLLAMA_BIN = shutil.which("ollama")
OLLAMA_ENV = os.environ.copy()
OLLAMA_ENV.update({
    "OLLAMA_HOST": "127.0.0.1:11434",
    "OLLAMA_MODELS": str(MODELS_DIR),
    "OLLAMA_KEEP_ALIVE": "-1",
    "OLLAMA_FLASH_ATTENTION": "1",
    "OLLAMA_KV_CACHE_TYPE": "q8_0",
    "OLLAMA_NUM_PARALLEL": "1",
    "OLLAMA_MAX_LOADED_MODELS": "2",
    "OLLAMA_SCHED_SPREAD": "1",      # দুই GPU-তে সমানভাবে ভাগ
    "OLLAMA_LOAD_TIMEOUT": "15m",
    "OLLAMA_NOPRUNE": "1",
})
print("models dir:", MODELS_DIR)
!df -h {base} | tail -1

## ৩. Ollama সার্ভার চালু

In [4]:
def ollama_ready():
    try:
        urllib.request.urlopen(f"{OLLAMA_URL}/api/tags", timeout=2).close(); return True
    except Exception:
        return False

if not ollama_ready():
    log = open("/tmp/ollama-server.log", "a")
    ollama_proc = subprocess.Popen([OLLAMA_BIN, "serve"], env=OLLAMA_ENV, stdout=log, stderr=subprocess.STDOUT)
    for _ in range(60):
        if ollama_ready(): break
        time.sleep(1)
    else:
        raise RuntimeError("Ollama চালু হয়নি। /tmp/ollama-server.log দেখুন")
print("✅ Ollama ready")

✅ Ollama ready


## ৪. মডেল ডাউনলোড (প্রায় ১৭ GB)

In [5]:
r = subprocess.run([OLLAMA_BIN, "pull", SOURCE_MODEL], env=OLLAMA_ENV)
if r.returncode != 0:
    print("সাধারণ pull ব্যর্থ, --insecure দিয়ে আবার চেষ্টা...")
    subprocess.run([OLLAMA_BIN, "pull", "--insecure", SOURCE_MODEL], env=OLLAMA_ENV, check=True)
subprocess.run([OLLAMA_BIN, "list"], env=OLLAMA_ENV)

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest 
pulling 4c5e2db039e9:   0% ▕                  ▏  33 KB/ 16 GB                  pulling manifest 
pulling 4c5e2db039e9:   0% ▕                  ▏ 1.5 MB/ 16 GB                  pulling manifest 
pulling 4c5e2db039e9:   0% ▕                  ▏ 7.5 MB/ 16 GB                  pulling manifest 
pulling 4c5e2db039e9:   0% ▕                  ▏  64 MB/ 16 GB                  pulling manifest 
pulling 4c5e2db039e9:   1% ▕                  ▏ 131 MB/ 16 GB                  pulling manifest 
pulling 4c5e2db039e9:   1% ▕                  ▏ 159 MB/ 16 GB                  pulling manifest 
pulling 4c5e2db039e9:   1% ▕                  ▏ 227 MB/ 16 GB                  pulling manifest 
pulling 4c5e2db039e9:   2% ▕                  ▏ 260 MB/ 16 GB                  pulling manifest 
pulling 4c5e2db039e9:   2% ▕       

NAME                                                        ID              SIZE     MODIFIED               
hf.co/JonathanColetti/Qwen3.8-27B-Uncensored-GGUF:Q4_K_M    2859adf8326a    17 GB    Less than a second ago    


CompletedProcess(args=['/usr/local/bin/ollama', 'list'], returncode=0)

## ৫. মডেলের ছবি দেখার ক্ষমতা আছে কিনা পরীক্ষা

না থাকলে ছোট একটা সহায়ক ভিশন মডেল (~৩ GB) নামবে, যে ছবি বর্ণনা করে দেবে আর মূল মডেল সেই বর্ণনা পড়ে উত্তর দেবে।

In [7]:
def api(path, payload, timeout=600):
    req = urllib.request.Request(f"{OLLAMA_URL}{path}", data=json.dumps(payload).encode(),
                                 headers={"Content-Type": "application/json"}, method="POST")
    with urllib.request.urlopen(req, timeout=timeout) as resp:
        return json.loads(resp.read())

caps = api("/api/show", {"model": SOURCE_MODEL}).get("capabilities") or []
print("মডেলের capabilities:", caps)
NATIVE_VISION = "vision" in caps
if NATIVE_VISION:
    print("✅ মডেল নিজেই ছবি বুঝতে পারে")
    HELPER = ""
else:
    print("ℹ️ মডেলে ছবি-সাপোর্ট নেই, সহায়ক ভিশন মডেল নামাচ্ছি:", VISION_HELPER)
    subprocess.run([OLLAMA_BIN, "pull", VISION_HELPER], env=OLLAMA_ENV, check=True)
    HELPER = VISION_HELPER

মডেলের capabilities: ['tools', 'thinking', 'completion', 'vision']
✅ মডেল নিজেই ছবি বুঝতে পারে


## ৬. মডেলকে আগে থেকেই GPU-তে লোড করে পিন করুন (এটাই দ্রুত রিপ্লাইয়ের চাবিকাঠি)

এই সেল নিজে থেকে সবচেয়ে বড় কনটেক্সট বেছে নেয় যেটায় মডেল **১০০% GPU**-তে ঢোকে। CPU-তে কিছু নামলে রিপ্লাই খুব ধীর হয়, তাই এটা জরুরি।

In [ ]:
def gpu_ps():
    return subprocess.run([OLLAMA_BIN, "ps"], env=OLLAMA_ENV, capture_output=True, text=True).stdout

def unload_all():
    for mdl in (SOURCE_MODEL,):
        try: api("/api/generate", {"model": mdl, "keep_alive": 0}, timeout=120)
        except Exception: pass
    time.sleep(2)

candidates = [NUM_CTX] + [c for c in NUM_CTX_FALLBACKS if c < NUM_CTX]
chosen = None
for ctx in candidates:
    unload_all()
    t = time.time()
    api("/api/generate", {"model": SOURCE_MODEL, "prompt": "hi", "stream": False, "keep_alive": -1,
                          "think": False, "options": {"num_ctx": ctx, "num_predict": 4}})
    print(f"num_ctx={ctx}: লোড হতে লাগল {time.time()-t:.0f}s")
    ps = gpu_ps(); print(ps)
    if "100% GPU" in ps:
        chosen = ctx; break
    print(f"⚠️ {ctx}-এ পুরোটা GPU-তে ঢোকেনি, ছোট কনটেক্সট চেষ্টা করছি…")

if chosen is None:
    chosen = candidates[-1]
    print("⚠️ কোনোটাতেই ১০০% GPU হয়নি; সবচেয়ে ছোটটা ব্যবহার হচ্ছে:", chosen)
NUM_CTX = chosen
print(f"✅ ব্যবহৃত NUM_CTX = {NUM_CTX} (মডেল GPU-তে পিন করা, keep_alive=-1)")
!nvidia-smi --query-gpu=index,memory.used,memory.total --format=csv

# গতি মাপা (এই ওয়ার্ম-আপ রিকোয়েস্টে মডেল পুরো গরম হয়ে যায়)
res = api("/api/generate", {"model": SOURCE_MODEL, "stream": False, "keep_alive": -1, "think": False,
    "prompt": "Explain in about 150 words why GPUs are good for running LLMs.",
    "options": {"num_ctx": NUM_CTX, "temperature": 0}})
print(f"গতি: {res['eval_count']/(res['eval_duration']/1e9):.1f} token/s")


Apps Shell****

In [13]:
import os
os.makedirs("/kaggle/working/chatapp", exist_ok=True)

## ৭. ওয়েব অ্যাপের ফাইল তৈরি

In [ ]:
%%writefile /kaggle/working/chatapp/index.html
<!DOCTYPE html>
<html lang="bn">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1, viewport-fit=cover">
<meta name="theme-color" content="#0b0b1a">
<title>Qwen Chat</title>
<style>
:root{--bg:#0b0b1a;--bg2:#12122a;--card:#1a1a38;--card2:#23234a;--bd:#2f2f5c;--fg:#f1f1ff;--mut:#9a9ac4;
--g1:#7c5cff;--g2:#22d3ee;--g3:#ff5ca8;--ok:#34d399;--grad:linear-gradient(135deg,var(--g1),var(--g2));--grad2:linear-gradient(135deg,var(--g3),var(--g1))}
*{box-sizing:border-box;-webkit-tap-highlight-color:transparent}
html,body{height:100%;margin:0;background:var(--bg);color:var(--fg);font-family:system-ui,-apple-system,"Noto Sans Bengali",Roboto,sans-serif}
body::before{content:"";position:fixed;inset:0;pointer-events:none;background:radial-gradient(600px 300px at 90% -5%,rgba(124,92,255,.28),transparent),radial-gradient(500px 300px at -10% 100%,rgba(34,211,238,.18),transparent)}
#app{display:flex;flex-direction:column;height:100dvh;position:relative}
header{display:flex;align-items:center;gap:8px;padding:10px 12px;padding-top:max(10px,env(safe-area-inset-top));background:rgba(11,11,26,.8);backdrop-filter:blur(12px);border-bottom:1px solid var(--bd);z-index:5}
button{font:inherit;color:var(--fg);background:var(--card);border:1px solid var(--bd);border-radius:12px;padding:8px 12px;cursor:pointer;transition:transform .08s,background .15s}
button:active{transform:scale(.94)}
#menu{width:40px;height:40px;padding:0;display:grid;place-items:center;flex:none}
svg.i{width:20px;height:20px;fill:none;stroke:currentColor;stroke-width:2;stroke-linecap:round;stroke-linejoin:round}
.brand{display:flex;align-items:center;gap:8px;min-width:0;flex:1}
.logo{width:34px;height:34px;border-radius:11px;flex:none;display:grid;place-items:center;font-weight:800;color:#fff;background:var(--grad);box-shadow:0 4px 16px rgba(124,92,255,.5)}
.names{min-width:0;line-height:1.15}
.names h1{font-size:15.5px;margin:0;font-weight:700;white-space:nowrap;overflow:hidden;text-overflow:ellipsis;background:var(--grad);-webkit-background-clip:text;background-clip:text;color:transparent}
.names small{font-size:11px;color:var(--mut);display:flex;align-items:center;gap:4px}
.dot{width:6px;height:6px;border-radius:50%;background:var(--ok);box-shadow:0 0 8px var(--ok)}
.badge{font-size:11px;padding:1px 7px;border-radius:999px;background:var(--card2);border:1px solid var(--bd);color:var(--mut)}
.tg{display:inline-flex;align-items:center;gap:6px;text-decoration:none;color:#fff;font-size:13px;font-weight:600;background:linear-gradient(135deg,#2AABEE,#229ED9);border-radius:999px;padding:7px 11px;flex:none;box-shadow:0 2px 10px rgba(34,158,217,.4)}
.tg svg{width:16px;height:16px;fill:#fff}
.think{display:flex;align-items:center;gap:6px;font-size:13px;color:var(--mut);cursor:pointer;flex:none;user-select:none}
.think input{display:none}
.sw{width:36px;height:21px;border-radius:999px;background:var(--card2);border:1px solid var(--bd);position:relative;transition:.2s}
.sw::after{content:"";position:absolute;top:2px;left:2px;width:15px;height:15px;border-radius:50%;background:#9aa3b8;transition:.2s}
.think input:checked + .sw{background:var(--grad);border-color:transparent}
.think input:checked + .sw::after{transform:translateX(15px);background:#fff}
#wrap{flex:1;position:relative;min-height:0}
#msgs{height:100%;overflow-y:auto;padding:14px 12px 20px;display:flex;flex-direction:column;gap:12px;overscroll-behavior:contain}
#empty{position:absolute;inset:0;display:flex;flex-direction:column;align-items:center;justify-content:center;gap:10px;text-align:center;padding:24px;pointer-events:none}
#empty .big{width:72px;height:72px;border-radius:24px;display:grid;place-items:center;font-size:32px;font-weight:800;color:#fff;background:var(--grad);box-shadow:0 10px 40px rgba(124,92,255,.55);animation:fl 3s ease-in-out infinite}
@keyframes fl{50%{transform:translateY(-6px)}}
#empty h2{margin:6px 0 0;font-size:20px}
#empty p{margin:0;color:var(--mut);font-size:14px}
.m{max-width:92%;padding:11px 14px;border-radius:18px;line-height:1.6;font-size:15.5px;overflow-wrap:anywhere;animation:pop .2s ease-out}
@keyframes pop{from{opacity:0;transform:translateY(8px)}to{opacity:1;transform:none}}
.u{align-self:flex-end;background:var(--grad2);border-bottom-right-radius:6px;box-shadow:0 4px 18px rgba(255,92,168,.25)}
.a{align-self:flex-start;background:var(--card);border:1px solid var(--bd);border-bottom-left-radius:6px;border-left:3px solid var(--g2)}
.m img{max-width:100%;max-height:220px;border-radius:12px;display:block;margin-bottom:6px}
pre{background:#07071a;border:1px solid var(--bd);border-radius:12px;padding:10px;overflow-x:auto;margin:8px 0}
code{font-family:ui-monospace,Menlo,Consolas,monospace;font-size:13.5px}
.m code{background:#07071a;padding:1px 6px;border-radius:6px;color:#9be9ff}
pre code{background:none;padding:0;color:#e8e8ff}
details{color:var(--mut);font-size:13.5px;margin-bottom:8px;background:var(--card2);border:1px solid var(--bd);border-radius:12px;padding:6px 10px}
summary{cursor:pointer}
.stat{color:var(--mut);font-size:12px;margin-top:6px}
.typing{display:inline-flex;gap:5px;padding:4px 0}
.typing i{width:8px;height:8px;border-radius:50%;background:var(--grad);animation:bl 1.2s infinite}
.typing i:nth-child(2){animation-delay:.15s}.typing i:nth-child(3){animation-delay:.3s}
@keyframes bl{0%,80%,100%{opacity:.25;transform:translateY(0)}40%{opacity:1;transform:translateY(-4px)}}
#down{position:absolute;right:14px;bottom:12px;width:40px;height:40px;padding:0;border-radius:50%;display:none;place-items:center;background:var(--grad);border:0;box-shadow:0 4px 16px rgba(124,92,255,.6);color:#fff}
.chip{display:inline-flex;align-items:center;gap:6px;background:var(--card);border:1px solid var(--bd);border-radius:999px;padding:4px 10px;font-size:13px;margin:0 6px 6px 0}
.chip b{cursor:pointer;color:#ff7b9c}
footer{border-top:1px solid var(--bd);padding:8px 10px;padding-bottom:max(8px,env(safe-area-inset-bottom));background:rgba(11,11,26,.85);backdrop-filter:blur(12px)}
#chips{display:flex;flex-wrap:wrap}
.row{display:flex;gap:8px;align-items:flex-end}
.row button{height:46px;display:grid;place-items:center;padding:0 12px}
#attach{width:46px;padding:0}
textarea{flex:1;resize:none;max-height:160px;min-height:46px;background:var(--card);color:var(--fg);border:1px solid var(--bd);border-radius:16px;padding:12px 14px;font:inherit;font-size:16px;outline:none;transition:border-color .15s,box-shadow .15s}
textarea:focus{border-color:var(--g1);box-shadow:0 0 0 3px rgba(124,92,255,.25)}
#send{background:var(--grad);border:0;width:50px;padding:0;color:#fff;box-shadow:0 4px 16px rgba(124,92,255,.5)}
#stop{background:linear-gradient(135deg,#ff5c7a,#d6264a);border:0;display:none;padding:0 16px}
/* drawer */
#shade{position:fixed;inset:0;background:rgba(0,0,0,.55);opacity:0;pointer-events:none;transition:.2s;z-index:20}
#drawer{position:fixed;top:0;bottom:0;left:0;width:min(82vw,320px);background:var(--bg2);border-right:1px solid var(--bd);transform:translateX(-102%);transition:transform .22s;z-index:21;display:flex;flex-direction:column;padding-top:env(safe-area-inset-top)}
body.open #shade{opacity:1;pointer-events:auto}
body.open #drawer{transform:none}
#drawer .top{padding:12px;display:flex;gap:8px;align-items:center}
#new{flex:1;background:var(--grad);border:0;font-weight:700;color:#fff;box-shadow:0 4px 16px rgba(124,92,255,.45)}
#list{flex:1;overflow-y:auto;padding:4px 10px 16px;display:flex;flex-direction:column;gap:6px}
.ci{display:flex;align-items:center;gap:6px;padding:10px 12px;border-radius:12px;background:var(--card);border:1px solid transparent;cursor:pointer}
.ci.on{border-color:var(--g1);background:var(--card2)}
.ci span{flex:1;min-width:0;white-space:nowrap;overflow:hidden;text-overflow:ellipsis;font-size:14px}
.ci b{color:#ff7b9c;padding:2px 6px;border-radius:8px}
.none{color:var(--mut);font-size:13px;text-align:center;padding:20px}
</style>
</head>
<body>
<div id="shade"></div>
<aside id="drawer">
  <div class="top"><button id="new">＋ নতুন চ্যাট</button><button id="close" style="width:40px;height:40px;padding:0">✕</button></div>
  <div id="list"></div>
</aside>
<div id="app">
  <header>
    <button id="menu" aria-label="চ্যাট তালিকা"><svg class="i" viewBox="0 0 24 24"><path d="M4 6h16M4 12h16M4 18h16"/></svg></button>
    <div class="brand">
      <div class="logo">Q</div>
      <div class="names">
        <h1 id="title">Qwen</h1>
        <small><span class="dot"></span><span>অনলাইন</span><span class="badge" id="vis" style="display:none">👁 ছবি</span></small>
      </div>
    </div>
    <a class="tg" href="https://t.me/rootmyphon" target="_blank" rel="noopener noreferrer" aria-label="Telegram">
      <svg viewBox="0 0 24 24"><path d="M9.78 15.67l-.4 4.24c.57 0 .82-.24 1.12-.53l2.69-2.57 5.57 4.08c1.02.57 1.74.27 2-.94l3.62-16.95c.34-1.55-.56-2.15-1.56-1.78L1.5 10.53c-1.51.59-1.49 1.43-.26 1.8l5.45 1.7L19.4 6.04c.6-.37 1.14-.17.69.2z"/></svg><span>Telegram</span>
    </a>
    <label class="think"><input type="checkbox" id="think"><span class="sw"></span><span>Think</span></label>
  </header>
  <div id="wrap">
    <div id="msgs"></div>
    <div id="empty">
      <div class="big">Q</div>
      <h2 id="hello">কী জানতে চান?</h2>
      <p>মেসেজ লিখুন, অথবা ছবি/ফাইল পাঠান</p>
    </div>
    <button id="down" aria-label="নিচে যান"><svg class="i" viewBox="0 0 24 24"><path d="M12 5v14M5 12l7 7 7-7"/></svg></button>
  </div>
  <footer>
    <div id="chips"></div>
    <div class="row">
      <button id="attach" title="ছবি/ফাইল"><svg class="i" viewBox="0 0 24 24"><path d="M21.4 11.1l-9.2 9.2a6 6 0 0 1-8.5-8.5l9.2-9.2a4 4 0 0 1 5.7 5.7l-9.2 9.2a2 2 0 0 1-2.8-2.8l8.5-8.5"/></svg></button>
      <textarea id="inp" rows="1" placeholder="মেসেজ লিখুন…"></textarea>
      <button id="send" title="পাঠান"><svg class="i" viewBox="0 0 24 24"><path d="M22 2L11 13"/><path d="M22 2l-7 20-4-9-9-4 20-7z"/></svg></button>
      <button id="stop">থামান</button>
    </div>
    <input type="file" id="file" multiple hidden>
  </footer>
</div>
<script>
const $=id=>document.getElementById(id);
const KEY='qwen_chats_v1';
let chats=[],cur=null,pending=[],busy=false,ctrl=null,stick=true;
try{chats=JSON.parse(localStorage.getItem(KEY)||'[]');}catch(e){chats=[];}
function save(){try{localStorage.setItem(KEY,JSON.stringify(chats.slice(0,40)));}catch(e){
  try{chats.forEach(c=>c.messages.forEach(m=>{delete m.images;}));localStorage.setItem(KEY,JSON.stringify(chats.slice(0,40)));}catch(_){}}}
const uid=()=>Date.now().toString(36)+Math.random().toString(36).slice(2,6);

function shortName(full){let n=(full||'').split('/').pop().split(':')[0];
  n=n.replace(/[-_]?(uncensored|abliterated|instruct|gguf|chat)/ig,'').replace(/[-_]+$/,'');
  n=n.replace(/-(\d+(\.\d+)?B)$/i,' $1');return n||full||'Qwen';}
fetch('/api/info').then(r=>r.json()).then(j=>{const s=shortName(j.model);$('title').textContent=s;$('hello').textContent=s+'-এর সাথে চ্যাট করুন';if(j.vision)$('vis').style.display='';}).catch(()=>{});

function esc(s){return s.replace(/&/g,'&amp;').replace(/</g,'&lt;').replace(/>/g,'&gt;');}
function md(src){return src.split('```').map((p,i)=>{
  if(i%2===1){const nl=p.indexOf('\n');const code=nl>=0?p.slice(nl+1):p;return '<pre><code>'+esc(code.replace(/\n$/,''))+'</code></pre>';}
  let h=esc(p);h=h.replace(/`([^`\n]+)`/g,'<code>$1</code>').replace(/\*\*([^*\n]+)\*\*/g,'<b>$1</b>').replace(/^#{1,6}\s+(.*)$/gm,'<b>$1</b>');
  return h.replace(/\n/g,'<br>');}).join('');}

/* scroll: শুধু তখনই নিচে যাবে যখন আপনি নিচেই আছেন */
const msgs=$('msgs');
function nearBottom(){return msgs.scrollHeight-msgs.scrollTop-msgs.clientHeight<80;}
msgs.addEventListener('scroll',()=>{stick=nearBottom();$('down').style.display=stick?'none':'grid';},{passive:true});
function toBottom(force){if(force||stick){msgs.scrollTop=msgs.scrollHeight;stick=true;$('down').style.display='none';}}
$('down').onclick=()=>toBottom(true);
function updateEmpty(){$('empty').style.display=msgs.children.length?'none':'flex';}
function addBubble(cls){const d=document.createElement('div');d.className='m '+cls;msgs.appendChild(d);updateEmpty();return d;}

function renderList(){
  const l=$('list');l.innerHTML='';
  if(!chats.length){l.innerHTML='<div class="none">এখনো কোনো চ্যাট নেই</div>';return;}
  chats.forEach(c=>{const d=document.createElement('div');d.className='ci'+(cur&&c.id===cur.id?' on':'');
    d.innerHTML='<span>💬 '+esc(c.title||'নতুন চ্যাট')+'</span><b data-d="1">✕</b>';
    d.onclick=e=>{if(e.target.dataset.d){if(confirm('এই চ্যাট মুছবেন?')){chats=chats.filter(x=>x.id!==c.id);if(cur&&cur.id===c.id)newChat();save();renderList();}return;}openChat(c.id);};
    l.appendChild(d);});
}
function drawMsgs(){
  msgs.innerHTML='';
  (cur?cur.messages:[]).forEach(m=>{
    const b=addBubble(m.role==='user'?'u':'a');
    if(m.role==='user'){(m.images||[]).forEach(d=>{const im=document.createElement('img');im.src='data:image/jpeg;base64,'+d;b.appendChild(im);});
      const t=document.createElement('div');t.innerHTML=esc(m.shown!==undefined?m.shown:m.content).replace(/\n/g,'<br>');b.appendChild(t);}
    else b.innerHTML=md(m.content);});
  updateEmpty();toBottom(true);
}
function newChat(){if(busy&&ctrl)ctrl.abort();cur=null;pending=[];renderChips();drawMsgs();renderList();closeDrawer();}
function openChat(id){if(busy&&ctrl)ctrl.abort();cur=chats.find(c=>c.id===id)||null;pending=[];renderChips();drawMsgs();renderList();closeDrawer();}
function openDrawer(){renderList();document.body.classList.add('open');}
function closeDrawer(){document.body.classList.remove('open');}
$('menu').onclick=openDrawer;$('shade').onclick=closeDrawer;$('close').onclick=closeDrawer;$('new').onclick=newChat;

function renderChips(){$('chips').innerHTML='';
  pending.forEach((a,i)=>{const c=document.createElement('span');c.className='chip';
    c.innerHTML=(a.kind==='image'?'🖼 ':'📄 ')+esc(a.name)+' <b data-i="'+i+'">✕</b>';$('chips').appendChild(c);});
  document.querySelectorAll('.chip b').forEach(b=>b.onclick=()=>{pending.splice(+b.dataset.i,1);renderChips();});}
function readImage(file){return new Promise((res,rej)=>{const url=URL.createObjectURL(file);const img=new Image();
  img.onload=()=>{const max=1280;let w=img.width,h=img.height;if(Math.max(w,h)>max){const s=max/Math.max(w,h);w=Math.round(w*s);h=Math.round(h*s);}
    const c=document.createElement('canvas');c.width=w;c.height=h;c.getContext('2d').drawImage(img,0,0,w,h);URL.revokeObjectURL(url);
    res(c.toDataURL('image/jpeg',0.85).split(',')[1]);};
  img.onerror=()=>rej(new Error('ছবি পড়া যায়নি'));img.src=url;});}
async function addFile(file){try{
  if(file.type.startsWith('image/')){pending.push({kind:'image',name:file.name,data:await readImage(file)});}
  else{const fd=new FormData();fd.append('file',file);const r=await fetch('/api/extract',{method:'POST',body:fd});const j=await r.json();
    if(!r.ok)throw new Error(j.error||'ফাইল পড়া যায়নি');pending.push({kind:'file',name:j.name,text:j.text,truncated:j.truncated});}
  }catch(e){alert(file.name+': '+e.message);}renderChips();}
$('attach').onclick=()=>$('file').click();
$('file').onchange=async e=>{for(const f of e.target.files)await addFile(f);e.target.value='';};
$('inp').addEventListener('input',e=>{e.target.style.height='auto';e.target.style.height=Math.min(e.target.scrollHeight,160)+'px';});
$('stop').onclick=()=>{if(ctrl)ctrl.abort();};
$('send').onclick=send;

async function send(){
  if(busy)return;
  const text=$('inp').value.trim();
  if(!text&&!pending.length)return;
  let content=text;const images=[];const names=[];
  for(const a of pending){
    if(a.kind==='image')images.push(a.data);
    else{content+='\n\n[সংযুক্ত ফাইল: '+a.name+(a.truncated?' (ছাঁটা)':'')+']\n```\n'+a.text+'\n```';names.push(a.name);}
  }
  if(!text)content=(images.length?'এই ছবিটি বিশ্লেষণ করো।':'এই ফাইলটি বিশ্লেষণ করো।')+content;
  if(!cur){cur={id:uid(),title:(text||names[0]||'ছবি').slice(0,40),messages:[]};chats.unshift(cur);}
  const msg={role:'user',content,shown:(text||'')+(names.length?'\n📄 '+names.join(', '):'')};
  if(images.length)msg.images=images;
  cur.messages.push(msg);
  const me=cur;
  stick=true;
  const ub=addBubble('u');
  images.forEach(d=>{const im=document.createElement('img');im.src='data:image/jpeg;base64,'+d;ub.appendChild(im);});
  const t=document.createElement('div');t.innerHTML=esc(msg.shown).replace(/\n/g,'<br>');ub.appendChild(t);
  $('inp').value='';$('inp').style.height='auto';pending=[];renderChips();toBottom(true);
  const ab=addBubble('a');ab.innerHTML='<span class="typing"><i></i><i></i><i></i></span>';toBottom(true);
  busy=true;$('send').style.display='none';$('stop').style.display='';
  ctrl=new AbortController();
  let answer='',thinking='',status='',stat='',raf=0;
  const paint=()=>{raf=0;
    if(cur!==me)return;
    let h='';
    if(thinking)h+='<details><summary>চিন্তা</summary>'+esc(thinking).replace(/\n/g,'<br>')+'</details>';
    h+=answer?md(answer):(status?'<span class="stat">'+esc(status)+'</span>':(thinking?'':'<span class="typing"><i></i><i></i><i></i></span>'));
    if(stat)h+='<div class="stat">'+stat+'</div>';
    ab.innerHTML=h;toBottom(false);};
  const draw=()=>{if(!raf)raf=requestAnimationFrame(paint);};
  const history=me.messages.map(m=>{const o={role:m.role,content:m.content};if(m.images)o.images=m.images;return o;});
  try{
    const res=await fetch('/api/chat',{method:'POST',signal:ctrl.signal,headers:{'Content-Type':'application/json'},
      body:JSON.stringify({messages:history,think:$('think').checked})});
    if(!res.ok)throw new Error('সার্ভার ত্রুটি '+res.status);
    const reader=res.body.getReader();const dec=new TextDecoder();let buf='';
    while(true){
      const {done,value}=await reader.read();if(done)break;
      buf+=dec.decode(value,{stream:true});
      let i;
      while((i=buf.indexOf('\n'))>=0){
        const line=buf.slice(0,i).trim();buf=buf.slice(i+1);if(!line)continue;
        let j;try{j=JSON.parse(line);}catch(e){continue;}
        if(j.status)status=j.status;
        if(j.error)answer+='\n⚠️ '+j.error;
        if(j.message){if(j.message.thinking)thinking+=j.message.thinking;if(j.message.content)answer+=j.message.content;}
        if(j.done&&j.eval_count&&j.eval_duration)stat=(j.eval_count/(j.eval_duration/1e9)).toFixed(1)+' token/s';
        draw();
      }
    }
  }catch(e){if(e.name==='AbortError')answer+=' ⏹';else answer+='\n⚠️ '+e.message;draw();}
  if(raf){cancelAnimationFrame(raf);}paint();
  me.messages.push({role:'assistant',content:answer});
  save();renderList();
  busy=false;ctrl=null;$('send').style.display='';$('stop').style.display='none';
}
drawMsgs();renderList();
</script>
</body>
</html>


In [ ]:
%%writefile /kaggle/working/chatapp/server.py
import hashlib
import io
import json
import os

import httpx
from fastapi import FastAPI, File, Request, UploadFile
from fastapi.responses import HTMLResponse, JSONResponse, PlainTextResponse, StreamingResponse

OLLAMA = "http://127.0.0.1:11434"
MODEL = os.environ["CHAT_MODEL"]
VISION_HELPER = os.environ.get("VISION_HELPER", "")
TOKEN = os.environ["APP_TOKEN"]
NUM_CTX = int(os.environ.get("NUM_CTX", "32768"))
SYSTEM = "You are a helpful assistant. Always reply in the same language the user writes in."

HERE = os.path.dirname(os.path.abspath(__file__))
HTML = open(os.path.join(HERE, "index.html"), encoding="utf-8").read()

app = FastAPI()
_vision = {"v": None}
_desc = {}


@app.middleware("http")
async def auth(request: Request, call_next):
    qk = request.query_params.get("k")
    k = qk or request.cookies.get("k")
    if k != TOKEN:
        return PlainTextResponse("Unauthorized", status_code=401)
    resp = await call_next(request)
    if qk == TOKEN:
        resp.set_cookie("k", TOKEN, max_age=3 * 86400, httponly=True, samesite="lax")
    return resp


async def has_vision():
    if _vision["v"] is None:
        try:
            async with httpx.AsyncClient(timeout=30) as c:
                r = (await c.post(f"{OLLAMA}/api/show", json={"model": MODEL})).json()
            _vision["v"] = "vision" in (r.get("capabilities") or [])
        except Exception:
            _vision["v"] = False
    return _vision["v"]


@app.get("/", response_class=HTMLResponse)
async def index():
    return HTML


@app.get("/api/info")
async def info():
    return {"model": MODEL, "vision": await has_vision() or bool(VISION_HELPER)}


@app.post("/api/extract")
async def extract(file: UploadFile = File(...)):
    name = file.filename or "file"
    data = await file.read()
    ext = name.lower().rsplit(".", 1)[-1] if "." in name else ""
    try:
        if ext == "pdf":
            from pypdf import PdfReader
            text = "\n".join((p.extract_text() or "") for p in PdfReader(io.BytesIO(data)).pages)
        elif ext == "docx":
            import docx
            d = docx.Document(io.BytesIO(data))
            text = "\n".join(p.text for p in d.paragraphs)
        elif ext in ("xlsx", "xlsm"):
            import openpyxl
            wb = openpyxl.load_workbook(io.BytesIO(data), read_only=True, data_only=True)
            lines = []
            for ws in wb.worksheets:
                lines.append(f"## {ws.title}")
                for row in ws.iter_rows(values_only=True):
                    lines.append("\t".join("" if v is None else str(v) for v in row))
            text = "\n".join(lines)
        else:
            text = data.decode("utf-8", errors="replace")
    except Exception as e:
        return JSONResponse({"error": str(e)}, status_code=400)
    limit = int(NUM_CTX * 2.2)
    return {"name": name, "text": text[:limit], "truncated": len(text) > limit}


@app.post("/api/chat")
async def chat(request: Request):
    body = await request.json()
    msgs = body.get("messages", [])
    think = bool(body.get("think", False))

    async def gen():
        vision = await has_vision()
        out = []
        async with httpx.AsyncClient(timeout=None) as c:
            # Images: send natively if the model has vision, otherwise describe them with a small helper model.
            for m in msgs:
                imgs = m.get("images") or []
                if imgs and not vision:
                    notes = []
                    for im in imgs:
                        h = hashlib.sha1(im.encode()).hexdigest()
                        if h not in _desc:
                            if not VISION_HELPER:
                                _desc[h] = "(no vision model available)"
                            else:
                                yield json.dumps({"status": "ছবি বিশ্লেষণ হচ্ছে…"}) + "\n"
                                r = await c.post(f"{OLLAMA}/api/chat", json={
                                    "model": VISION_HELPER, "stream": False, "keep_alive": "2m", "options": {"num_gpu": 0},
                                    "messages": [{"role": "user", "images": [im], "content":
                                        "Describe this image in detail. Transcribe all visible text exactly. "
                                        "Mention objects, people, layout, numbers and anything notable."}],
                                })
                                _desc[h] = r.json().get("message", {}).get("content", "")
                        notes.append(_desc[h])
                    extra = "\n\n".join(f"[Attached image {i + 1} - analysis by a vision model]\n{d}" for i, d in enumerate(notes))
                    m = {"role": m["role"], "content": m.get("content", "") + "\n\n" + extra}
                out.append(m)

            payload = {
                "model": MODEL,
                "messages": [{"role": "system", "content": SYSTEM}] + out,
                "stream": True,
                "keep_alive": -1,
                "think": think,
                "options": {"num_ctx": NUM_CTX, "temperature": 0.7},
            }
            for attempt in range(2):
                async with c.stream("POST", f"{OLLAMA}/api/chat", json=payload) as r:
                    if r.status_code != 200:
                        err = (await r.aread()).decode("utf-8", "replace")
                        if attempt == 0 and "think" in err.lower():
                            payload.pop("think", None)  # model does not support the think flag
                            continue
                        yield json.dumps({"error": err[:500]}) + "\n"
                        return
                    async for line in r.aiter_lines():
                        if line:
                            yield line + "\n"
                    return

    return StreamingResponse(gen(), media_type="application/x-ndjson",
                             headers={"Cache-Control": "no-cache", "X-Accel-Buffering": "no"})


## ৮. সার্ভার + পাবলিক লিঙ্ক চালু (এটাই শেষ ধাপ)

এই সেলটা চলতে থাকবে — **বন্ধ করবেন না**। নিচে লিঙ্ক ছাপা হলে ফোনের Chrome-এ খুলুন।

In [ ]:
import re, queue, threading

# cloudflared ইনস্টল
if not shutil.which("cloudflared"):
    !curl -fsSL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared
    !chmod +x /usr/local/bin/cloudflared

APP_TOKEN = secrets.token_urlsafe(16)
app_env = os.environ.copy()
app_env.update({"CHAT_MODEL": SOURCE_MODEL, "VISION_HELPER": HELPER, "APP_TOKEN": APP_TOKEN, "NUM_CTX": str(NUM_CTX)})

if "app_proc" in globals() and app_proc.poll() is None:
    app_proc.terminate(); time.sleep(1)
app_log = open("/tmp/chatapp.log", "w")
app_proc = subprocess.Popen(["python", "-m", "uvicorn", "server:app", "--host", "127.0.0.1", "--port", str(PORT)],
                            cwd="/kaggle/working/chatapp", env=app_env, stdout=app_log, stderr=subprocess.STDOUT)
for _ in range(30):
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{PORT}/api/info?k={APP_TOKEN}", timeout=2).close(); break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("ওয়েব সার্ভার চালু হয়নি: " + open("/tmp/chatapp.log").read()[-1500:])

if "tunnel_proc" in globals() and tunnel_proc.poll() is None:
    tunnel_proc.terminate(); time.sleep(1)
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--no-autoupdate", "--protocol", "http2", "--edge-ip-version", "4",
     "--url", f"http://127.0.0.1:{PORT}"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
q = queue.Queue()
threading.Thread(target=lambda: [q.put(l) for l in iter(tunnel_proc.stdout.readline, "")], daemon=True).start()
PUBLIC_URL, deadline = None, time.monotonic() + 60
while time.monotonic() < deadline and not PUBLIC_URL:
    try:
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", q.get(timeout=1))
        if m: PUBLIC_URL = m.group(0)
    except queue.Empty:
        pass
if not PUBLIC_URL:
    raise RuntimeError("লিঙ্ক পাওয়া যায়নি, সেলটা আবার চালান")

print("\n" + "=" * 60)
print("📱 ফোনের Chrome-এ এই লিঙ্কটা খুলুন:\n")
print(f"{PUBLIC_URL}/?k={APP_TOKEN}")
print("\n⚠️ লিঙ্কটা কাউকে দেবেন না — যার কাছে আছে সে আপনার AI ব্যবহার করতে পারবে।")
print("=" * 60)

# মডেলকে সবসময় GPU-তে গরম রাখা (আনলোড হলে ফেরত লোড করে)
def keep_warm():
    while True:
        try:
            api("/api/generate", {"model": SOURCE_MODEL, "keep_alive": -1, "options": {"num_ctx": NUM_CTX}}, timeout=900)
        except Exception:
            pass
        time.sleep(120)
threading.Thread(target=keep_warm, daemon=True).start()

# সেশন জাগিয়ে রাখা
while True:
    time.sleep(30)
    if app_proc.poll() is not None or tunnel_proc.poll() is not None:
        print("⚠️ সার্ভার বা টানেল বন্ধ হয়ে গেছে, এই সেলটা আবার চালান"); break